# Does the SBND DENT simulation improve data/MC agreement?

*A shape-only, no-fit study with the SBND release detector-access data. Generated 2026-10-07 from `plotting/dent_study/report_cells.py`.*

## Summary

**Question.** DENT is the detector variation that models the cathode-crossing (and related) effects in SBND. After the
cathode-crosser cut it is one of the larger uncorrelated systematics. Does simulating it (the DENT detvar sample, or the
production CV shifted by the DENT ratio) describe the data better than the CV does, and where does it match or fail to match?

**Method in one paragraph.** SBND release detector-access data (the file has no energy or momentum
column, and none is used) are compared with the SBND simulation, always area-normalised and shape-only, in each observable's own
binning, with shape chi-square and p-value as the metrics. Nothing is fitted: DENT enters only at fixed strengths 0, 0.5 and 1.0.
Tier A uses the detvar CV and DENT samples directly (MC statistics only). Tier B uses the production CV (`cvrw` plus `cvwgt`)
with the full systematic covariance. Tier C asks how well one observable could tell the two apart, using fake data.

**Findings.**

1. **The data do not show that DENT applied to the central value improves the agreement, and they do not reject it.** With the
   production covariance, shifting the CV by the full DENT effect changes the shape chi-square by less than 2 units for 9 of
   the 10 primary observables (5 improve, 5 worsen). The tenth, `muon_calo_csda`, is dominated by a covariance re-centering artefact
   (section 7). Across the 111 SBND regenerated detector-access channels the largest changes are +5.8 and -3.1 units.
2. **Tier A (MC statistics only) splits by physics.** The DENT sample is closer to the data for the seven calorimetry and
   particle-ID observables (largest for `muon_calo_csda`, 10,534 to 2,188 on 19 degrees of freedom), and farther for the three
   cathode-geometry observables (`crosser_flag`, `muon_abs_endx`, `muon_endx_full`; for `muon_abs_endx` 51 to 338 on 19 degrees of freedom).
   The calorimetry/PID improvement disappears as the other systematics are added to the covariance (section 6), so it is not
   evidence that DENT is the cause of those data/MC differences.
3. **The cathode features that make DENT large are only partly confirmed by the data.** The fraction of events flagged as crossers
   is 27.46 +- 0.18 % in data against 27.92 % (CV) and 25.36 % (DENT). Non-crossing muons ending within 5 cm of the cathode
   (the "pile-up") are 1.95 % of events in data against 2.07 % (CV) and 3.78 % (DENT): the data follow the CV, and the large DENT
   pile-up is not present. The deficit of crossing muons that end within 4 cm of the cathode is, however, in the direction DENT
   predicts (data 0.68 %, CV 1.14 %, DENT 0.55 %). These cathode comparisons were seen before the study plan was written, so they are
   hypothesis-prompted, not independent confirmations.
4. **DENT as an uncertainty is not redundant.** Removing DENT from the covariance makes the muon end-x observables fail
   (`muon_abs_endx`: p = 0.46 with DENT, 0.003 without; `muon_endx_full`: 0.15 against 2e-5). It absorbs a real data/MC difference.
5. **No single observable has the power to decide.** With pseudo-experiments through the fixed covariance, the separation between
   the "CV" and "full DENT" expectations is d' = 0.5 to 1.0 per observable, and all ten primaries have two-sided p >= 0.15 under all
   three truths (CV, half DENT, full DENT). The observables are correlated, so their p-values cannot be multiplied together.

**Recommendation.** Keep DENT as an uncertainty, and do not shift the central value by it on the strength of this study. The
mechanism that most plausibly makes DENT large at high energy is the near-cathode endpoint pile-up, which the data lack. A
near-cathode endpoint treatment (an endpoint cut, or a dedicated near-cathode observable) is the natural follow-up and is not part of this study.

**What this study cannot say.** It uses no energy or momentum quantity, so it cannot test the effect of DENT on the
reconstructed-energy distributions that drive the analysis. The detvar samples are neutrino-only and their CV is not the
production nominal. ICARUS has no DENT variation; its regenerated studies are reported only as a baseline.

## 1. Data, samples and rules

* **Data.** `spineosc_numu_sbnd_release-v1.1.0_detector_access_DONOTLOOK.root`, tree `events/sbnd_bnblight_full/selected`
  (area-normalised comparisons only; no event counts are quoted). This file is cleared for detector-level studies and has no energy or momentum branch. SBND data files that
  carry energy are blinded and were not opened.
* **Production MC (Tier B).** `spineosc_numu_sbnd_v1.1.0_wsyst_cvrw.root`, CV-reweighted (`cvwgt` in every beam-MC weight), with the full
  interaction-model, flux, geant4 and detector systematic suite.
* **Detvar MC (Tier A).** `sbnd_detvar_newcv` and `sbnd_detvar_newdent`. They have no `cvwgt` and no interaction-model
  systematics, so only MC statistics enter their covariance.
* **Selection.** At least one reconstructed proton, `reco_fiducialize_cathode == 1` and `reco_veto_sbnd_highy_highz == 1`. The MC also
  applies the flash-PE threshold, which is trigger emulation and is not applied to data.
* **Rules.** No fitting; every PROfit job uses `--area-norm` and `--shapeonly`; shape chi-square and p-value
  (`ndf = nbins - 1`) are the metrics; systematics are assessed in the observable's own binning; plots follow
  `plotting/data_access_results.ipynb`.
* **Observables.** Twenty-one energy-free observables were registered, binned by equal-population quantiles of the detvar CV only.
  The **primary set** (ten observables) was chosen by an MC-only criterion: the CV-versus-DENT shape chi-square at data-sized
  statistics, per degree of freedom, is at least 1. The analyst had already seen exploratory data comparisons (section 9), so the choice
  was not blind. Everything else is diagnostic.
* **Plan.** The plan and every change to it after the first result are in `docs/dent_study/PREREGISTRATION.md`.

In [ ]:
import sys, json, pathlib
sys.path.insert(0, "/Users/mueller/Projects/spineosc/plotting/dent_study")
import numpy as np, pandas as pd
import dentstudy as ds
from mutools.plotting import use_style
use_style('rootlike')
REPO = ds.REPO
FIG = REPO / "docs" / "dent_study" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
res = ds.load_results(); tc = ds.load_tierC()
import re
LABEL = {n: o["label"].replace("$", "").replace("\\", "").replace("_{", "_").replace("}", "") for n, o in ds.OBS.items()}
def lab(s): return LABEL.get(s, s.replace("_", " "))
PRIM = [p for p in ds.PRIMARY if p in ds.OBS]
print(f"{res.tag.nunique()} study runs loaded; primary set = {len(PRIM)} observables; Tier C throws loaded for {tc.study.nunique() if len(tc) else 0} studies")

## 2. Quality control
Every study is checked for four things: that `--shapeonly` is live (the chi-square must differ from a run without the flag,
because `ndf` is the same either way), that `ndf = nbins - 1`, that no DENT injection was silently ignored by PROfit (the
injected chi-square must differ from the plain one), and that every step finished.

In [ ]:
# Quality control per study: shape-only live, ndf = nbins - 1 (area-normalised), every step finished.
qc = res[res.variant == "v1"].groupby("tag").agg(det=("det", "first"), tier=("tier", "first"), live=("shapeonly_live", "first"),
        ndf_ok=("ndf", lambda s: True), n_rows=("chi2", "size")).reset_index()
bad_ndf = res[(res.variant == "v1") & (res.ndf != res.nbins - 1)]
print(f"studies with results: {len(qc)}  |  shape-only verified live (chi2 differs without the flag): {int(qc.live.fillna(False).sum())} of {len(qc)}")
print(f"rows where ndf != nbins-1 (would indicate no area normalisation): {len(bad_ndf)}")
summ = qc.groupby(["det", "tier"]).agg(studies=("tag", "size"), shapeonly_live=("live", lambda s: int(s.fillna(False).sum()))).reset_index()
ds.show_table(summ, "Studies by detector and tier")
# Audit: a DENT injection that PROfit silently ignored leaves chi2 identical to v1. List any such study.
w = res[res.variant.isin(["v1", "inj1p0"])].pivot_table(index=["tag", "channel"], columns="variant", values="chi2")
if "inj1p0" in w:
    inert = w[(w.v1 - w.inj1p0).abs() < 1e-9].reset_index().tag.unique().tolist()
    print(f"studies whose DENT injection left chi2 unchanged (invalid, must be re-run): {inert if inert else 'none'}")

## 3. Mechanism: what DENT changes, and whether the data show it
The figures compare the detvar CV, the detvar DENT sample and the data with one selection (the flash threshold is applied to the
MC only). Markers are data with statistical errors. They were made before the study plan was fixed, so the cathode
comparisons here are **hypothesis-prompted**.

* In the matched CV/DENT events, 4.5 % of the CV cathode crossers are no longer flagged as crossers in the
  DENT sample. 96 % of those are true crossers, and the reverse flips are about 2 % of the forward ones. The muon flag is the cause in 69 % of the flips.
  The median missing track length is 6.3 cm, but the mean is 30 cm and 18 % lose more than 50 cm.
* In the DENT sample 69 % of the flipped events end within 5 cm of the cathode (52 % in the CV), against 3.0 % of all
  non-crossers. This is the near-cathode pile-up.
* **Against the data:** the crosser fraction and the pile-up follow the CV, not DENT (summary, point 3). The deficit of crossing
  muons ending within 4 cm of the cathode is in DENT's direction, but the corresponding excess of non-crossing muons there is not present in data
  (data/CV = 0.91, DENT/CV = 2.0). DENT therefore appears to remove crossers correctly near the cathode and then to
  over-populate the non-crosser endpoint pile-up.

In [ ]:
for f, cap in (("mechanism_missing_length.png", "Track length missing from the flipped crossers: (a) distribution, (b) against the length beyond the cathode, (c) where the CV track ends, (d) probability of flipping."),
               ("mechanism_endpoint_cut.png", "(a) Distance of the muon endpoint from the cathode for the flipped events and for non-crossers. (b) DENT/CV ratio of selected events against reconstructed visible energy (MC only): without the crosser cut, with it, and with it plus |x_end| >= 5 cm. The extra cut leaves the ratio within 1.6 % of unity.")):
    ds.show_figure(FIG / f, cap)

## 4. Tier A: the detvar samples against the data

In [ ]:
A = res[(res.tier == "A") & (res.variant == "v1")]
cv = A[A.which == "cv"].set_index("study"); dn = A[A.which == "dent"].set_index("study")
ta = pd.DataFrame({"chi2_cv": cv.chi2, "chi2_dent": dn.chi2, "ndf": cv.ndf}).dropna()
ta["dchi2"] = ta.chi2_cv - ta.chi2_dent; ta["label"] = [lab(s) for s in ta.index]; ta["primary"] = ta.index.isin(PRIM)
ta = ta.sort_values("dchi2", ascending=False)
tab = ta.reset_index()[["study", "primary", "ndf", "chi2_cv", "chi2_dent", "dchi2"]]
ds.show_table(tab.rename(columns={"study": "observable", "chi2_cv": "chi2 (detvar CV)", "chi2_dent": "chi2 (detvar DENT)", "dchi2": "chi2(CV) - chi2(DENT)"}), "Tier A: detvar samples used directly (MC statistics only)")
fig = ds.plot_tierA(ta[ta.primary], savepath=FIG / "tierA_primary.png", title="Tier A (detvar CV vs DENT)")
ds.show_figure(FIG / "tierA_primary.png", "Tier A, primary observables. Bars right of zero: the DENT sample agrees better with the data.")

Reading the table: positive `chi2(CV) - chi2(DENT)` means the DENT sample is closer to the data. These use the detvar samples
with an MC-statistics-only covariance, so the numbers are large and the comparison is about shapes, not consistency. The detvar CV is also not
the production nominal, so part of each chi-square is a CV difference unrelated to DENT. The ordering is physical: the seven
calorimetry and particle-ID observables improve with DENT, and the three cathode-geometry observables
(`crosser_flag`, `muon_abs_endx`, `muon_endx_full`) get worse. The diagnostic observables are in the table only.

## 5. Tier B: production CV with the full systematic covariance
Prediction: `spineosc_numu_sbnd_v1.1.0_wsyst_cvrw.root` (`cvwgt` applied) with every systematic, area-normalised and shape-only.
Four variants of the same prediction: `v1` (standard, DENT in the covariance), `exDENT` (DENT removed from the covariance), and the CV
shifted by 0.5 or 1.0 times the DENT ratio (`inj0p5`, `inj1p0`) with DENT kept in the covariance. DENT is never fitted. The
shifted-CV and DENT-removed variants cannot be combined (PROfit then ignores the injection), so that combination is not shown.

**What the table says.**
* The production CV is already acceptable in six of the ten primaries (p = 0.11 to 0.59 for `crosser_flag`, `muon_proton_score`,
  `muon_abs_endx`, `muon_endx_full`, `proton_calo_csda`, `proton_muon_score`). It fails for `muon_calo_csda` (p = 1.7e-10),
  `proton_pion_score` (1.2e-8), `proton_mip_score` (2.2e-4) and `muon_pion_score` (0.010). Injecting DENT repairs none of them: for
  `proton_pion_score`, `proton_mip_score` and `muon_pion_score` the change is at most 1.4 units, and the `muon_calo_csda` change is an artefact (section 7).
* The change in chi-square from the full DENT shift is below 2 units in nine of ten primaries, with five improving and five worsening.
* **DENT as an uncertainty matters for the endpoint.** Removing it from the covariance raises chi-square for `muon_abs_endx` from 19.0 to 40.3
  (p 0.46 to 0.003), for `muon_endx_full` from 29.9 to 62.0 (p 0.15 to 2e-5), and for `muon_pion_score` from 30.4 to 41.0.

In [ ]:
B = res[(res.tier == "B") & (res.channel == 0) & res.variant.isin(list(ds.VARIANTS_B))]
tb = pd.DataFrame(columns=["observable", "dchi2 (0 vs 1)"]); tcs = pd.DataFrame()
if B.empty: print("no Tier B results yet")
wide = B.pivot_table(index="study", columns="variant", values=["chi2", "p"]); ndf = B[B.variant == "v1"].set_index("study").ndf
tb = pd.DataFrame({"ndf": ndf})
for v in ds.VARIANTS_B:
    if ("chi2", v) in wide.columns: tb[f"chi2 {v}"] = wide[("chi2", v)]; tb[f"p {v}"] = wide[("p", v)]
if "chi2 v1" in tb and "chi2 inj1p0" in tb: tb["dchi2 (0 vs 1)"] = tb["chi2 v1"] - tb["chi2 inj1p0"]
tb = tb.reindex([p for p in PRIM if p in tb.index] + [s for s in tb.index if s not in PRIM]).reset_index().rename(columns={"study": "observable"})
tb.insert(1, "primary", tb.observable.isin(PRIM))
if len(tb): ds.show_table(tb, "Tier B: production CV (cvrw + cvwgt), full systematics, area-normalised and shape-only")
prim_b = B[B.study.isin(PRIM)]
if len(prim_b):
    ds.plot_pvalues(prim_b, [p for p in PRIM if p in set(prim_b.study)], [lab(p) for p in PRIM if p in set(prim_b.study)], savepath=FIG / "tierB_pvalues_primary.png")
    ds.show_figure(FIG / "tierB_pvalues_primary.png", "Tier B shape-only p-values for the primary observables, four prediction variants.")

### Covariance ablation
The same comparison of strength 0 against 1.0, with the covariance built from progressively more systematics (the strength is never
fitted). The large gains for calorimetry and particle-ID observables with a statistics-plus-DENT covariance shrink as the other systematics are
added: for `proton_calo_csda` the change is +58.7 with statistics and DENT only, +3.0 with all detector systematics, and +0.8 with the full covariance.
The Tier A improvement for those observables is therefore not specific to DENT; other detector variations cover the same data/MC difference.
Several observables change sign with the covariance, which is itself a measure of how little the chi-square change can be relied on.

In [ ]:
# Covariance ablation: how much of the DENT effect survives as other systematics are added. All configurations keep DENT in the
# covariance (so the injection is valid) and compare theta = 0 with theta = 1.0. Fixed configurations only; nothing is fitted.
CFG = {"stat + DENT": ("abl_statDENT_t0", "abl_statDENT_t1"), "stat + all detector": ("abl_det_t0", "abl_det_t1"),
       "all but other detector": ("abl_nodetother_t0", "abl_nodetother_t1"), "full (standard)": ("v1", "inj1p0")}
Bc = res[(res.tier == "B") & (res.channel == 0)]
rows = []
for name in [p for p in PRIM if p in set(Bc.study)] + [s for s in sorted(set(Bc.study)) if s not in PRIM]:
    g = Bc[Bc.study == name].set_index("variant").chi2
    r = {"observable": name, "primary": name in PRIM}
    for cfg, (a, b) in CFG.items():
        if a in g.index and b in g.index: r[f"{cfg}: chi2(0)"] = g[a]; r[f"{cfg}: chi2(1)"] = g[b]; r[f"{cfg}: dchi2"] = g[a] - g[b]
    rows.append(r)
abl = pd.DataFrame(rows)
if "stat + DENT: dchi2" in abl:
    ds.show_table(abl[["observable", "primary"] + [c for c in abl.columns if c.endswith("dchi2")]], "Covariance ablation: dchi2 = chi2(theta=0) - chi2(theta=1) (positive: the DENT shift helps), by covariance content")
    import matplotlib.pyplot as plt
    from mutools.plotting.helpers import mark_axis
    ds.style(); pr = abl[abl.primary & abl["stat + DENT: dchi2"].notna()]
    if len(pr):
        fig, ax = plt.subplots(figsize=(11, 4.8)); x = np.arange(len(pr)); w = 0.2
        for i, (cfg, col) in enumerate(zip(CFG, ["C2", "C3", "C0", "C1"])):
            if f"{cfg}: dchi2" in pr: ax.bar(x + (i - 1.5) * w, pr[f"{cfg}: dchi2"], w, color=col, label=cfg)
        ax.axhline(0, color="0.3", lw=1); ax.set_yscale("symlog", linthresh=5); ax.set_xticks(x); ax.set_xticklabels([lab(o) for o in pr.observable], rotation=35, ha="right")
        ax.set_ylabel(r"$\chi^2(\theta{=}0)-\chi^2(\theta{=}1)$"); ax.legend(fontsize=9, ncol=2); mark_axis(ax, r"$\bf{SBND}$ Data Access", vadj=0.05)
        fig.savefig(FIG / "tierB_ablation.png", bbox_inches="tight", dpi=200); ds.show_figure(FIG / "tierB_ablation.png", "Tier B covariance ablation for the primary observables.")
else:
    print("ablation results not available yet")

### Panels
PROfit-style panels for the ten primary observables (production CV, and the CV shifted by full DENT; area-normalised, shape-only).

In [ ]:
# PROfit-style panels (mutools, same configuration as plotting/data_access_results.ipynb) for the primary observables
import glob
made = []
for name in PRIM:
    o = ds.OBS[name]; e = o["edges"]
    for variant, tag_ in (("v1", "CV"), ("inj1p0", "CV + DENT")):
        f = ds.RUN / f"dent_sbnd_run1_dd_{name}_{variant}_PROplot_histograms.root"
        if not f.exists(): continue
        p = ds.render_panel(f, FIG / "panels", f"{name}_{variant}", lab(name), xlim=[e[0], e[-1]]); made.append((name, variant, p))
for name, variant, p in made:
    ds.show_figure(p, f"{lab(name)}: " + ("production CV" if variant == "v1" else "production CV shifted by 1.0 x DENT") + " (area-normalised, shape-only)")
print(len(made), "panels rendered")

## 6. What the cathode and calorimetry results mean together
Tier A says the DENT sample matches the data's calorimetry and PID shapes better but its cathode geometry worse. Tier B says that, once the
other systematics are included, shifting the CV by DENT moves the chi-square by only a unit or two. These are consistent: the calorimetry/PID gain is covered by other detector
systematics, and the cathode-geometry loss is small in Tier B only because DENT is itself in the covariance. Section 7 asks the sharper question.

## 7. Is the DENT shift an improvement? Fixed-covariance analysis and pseudo-experiments
**Why PROfit's own change in chi-square is not enough.** With `--inject-systs-cv` PROfit re-derives the systematic covariance
around the *shifted* prediction. For `muon_calo_csda` the first-bin fractional DENT covariance falls from 0.84 to 0.51 and the
statistics-only chi-square from 14,606 to 3,556, so PROfit's chi-square at strength 1.0 mixes the central-value shift with a change of the
covariance. PROfit reports a change of -18.8 for that observable; with the covariance held at its strength-0 form it is +0.20.

**The replica.** A numpy replica of PROfit's shape-only chi-square, `chi2 = (n-p)^T [diag(n) + F o (p p^T)]^-1 (n-p)`, with `F` the
collapsed total fractional covariance and `p` the prediction area-normalised to the data, reproduces PROfit at all three strengths
(maximum difference printed below). Holding `F` at its strength-0 value for both predictions gives the **fixed-covariance change in chi-square**,
`chi2(0) - chi2(1)`; positive favours the DENT-shifted prediction. The same replica runs 20,000 pseudo-experiments per truth (Poisson
fluctuations plus Gaussian systematic throws through `F`), with truth at strength 0, 0.5 or 1.0, to calibrate it. The p-value is two-sided and
empirical, with a floor of 2/(N+1). No strength is estimated; the three truths are fixed points.

In [ ]:
import replica as rp
rs = pd.read_csv(REPO / "docs" / "dent_study" / "replica_summary.csv")
rs = rs.set_index("study").loc[[p for p in PRIM if p in set(rs.study)]]
dmax = 0.0
for name in rs.index:
    v = rp.validate(rp.load(name)); dmax = max(dmax, max(abs(x) for x in v.values()))
print(f"replica versus PROfit chi-square, largest absolute difference over {len(rs)} primaries and 3 strengths: {dmax:.3f}")
t = rs.reset_index().copy(); t.insert(1, "label", [lab(s) for s in t.study])
t["nearer truth CV?"] = (abs(t.dchi2_fixed_data - t.mean_0) < abs(t.dchi2_fixed_data - t.mean_1))
ds.show_table(t[["label", "profit_dchi2", "dchi2_fixed_data", "mean_0", "sd_0", "p2_0", "mean_0.5", "sd_0.5", "p2_0.5", "mean_1", "sd_1", "p2_1", "dprime", "nearer truth CV?"]].rename(columns={
    "profit_dchi2": "PROfit change", "dchi2_fixed_data": "fixed-cov change (data)", "mean_0": "mean|CV", "sd_0": "sd|CV", "p2_0": "p2|CV", "mean_0.5": "mean|half", "sd_0.5": "sd|half", "p2_0.5": "p2|half", "mean_1": "mean|DENT", "sd_1": "sd|DENT", "p2_1": "p2|DENT", "dprime": "d'"}),
    "Fixed-covariance chi2(0) - chi2(1) in data, against 20,000 pseudo-experiments for truth = CV, half DENT, full DENT (two-sided p)")
print(f"data nearer the CV-truth mean than the DENT-truth mean: {int(t['nearer truth CV?'].sum())} of {len(t)}; smallest two-sided p over all truths: {t[['p2_0','p2_0.5','p2_1']].min().min():.3f}; d' range {t.dprime.min():.2f} to {t.dprime.max():.2f}")
import matplotlib.pyplot as plt
from mutools.plotting.helpers import mark_axis
ds.style(); fig, ax = plt.subplots(figsize=(9.5, 0.55 * len(t) + 1.8)); y = np.arange(len(t))[::-1]
for off, col, key, nm in ((0.2, "C0", "0", "truth CV"), (0.0, "C2", "0.5", "truth half DENT"), (-0.2, "C1", "1", "truth full DENT")):
    ax.errorbar(t[f"mean_{key}"], y + off, xerr=t[f"sd_{key}"], fmt="o", color=col, ms=5, capsize=2, label=nm + " (mean +- 1 sd)")
ax.plot(t.dchi2_fixed_data, y, "k*", ms=11, label="data", zorder=5)
ax.axvline(0, color="0.4", lw=1); ax.set_yticks(y); ax.set_yticklabels(t.label)
ax.set_xlabel(r"fixed-covariance $\chi^2(\theta{=}0)-\chi^2(\theta{=}1)$" + "\n[positive: DENT-shifted prediction closer to data]"); ax.legend(fontsize=8, loc="lower right")
mark_axis(ax, r"$\bf{SBND}$ Data Access", vadj=0.05); fig.savefig(FIG / "replica_dchi2.png", bbox_inches="tight", dpi=200)
ds.show_figure(FIG / "replica_dchi2.png", "Data against the pseudo-experiment expectations for the three fixed truths (fixed covariance).")

**Reading the table and figure.**
* The spread of the fixed-covariance change under any one truth is 1.8 to 4.9 units, comparable to the separation between truths (d' = 0.5 to 1.0).
  A single observable therefore cannot establish either hypothesis.
* No primary observable rejects any truth: the smallest two-sided p is 0.155 (`muon_endx_full`, truth full DENT).
* Descriptively, the data lie nearer to the mean expected for truth CV than for truth full DENT in the count shown above. That count is a summary of the figure, not a test, and the observables are correlated.
* The ranking agrees with Tier A for the cathode-geometry observables (the data prefer the CV for `muon_abs_endx`, `muon_endx_full`) but not for the
  calorimetry/PID ones, where the production covariance already accommodates the data/MC difference.

## 8. Supplementary: PROfit's own pseudo-experiments (50 throws)
These are PROfit `--pseudo-experiment` throws (Poisson plus Gaussian nuisance throws) with the DENT strength injected into the fake data at 0, 0.5 or 1.0, then
the same chi-square change as in section 5. They are kept for completeness. Because PROfit re-centers the covariance on the injected prediction, they
carry the artefact described in section 7 (for example `muon_calo_csda` has a mean change of 162 with a standard deviation of 268, which is not meaningful).
The calibrated version is the replica analysis in section 7. Poisson-only fake data (an earlier version of this tier, replaced
on 2026-10-07) was too narrow to calibrate anything: its spread was 0.2 to 0.3 units against 1 to 3 for the pseudo-experiments.

In [ ]:
rows = []
for name in PRIM:
    sub = tc[tc.study == name] if len(tc) else tc
    row_b = tb[tb.observable == name] if "observable" in tb else tb.iloc[0:0]
    if len(sub) == 0 or len(row_b) == 0: continue
    d_obs = float(row_b["dchi2 (0 vs 1)"].iloc[0]); r = ds.tierC_pvalues(sub, d_obs); r.update(study=name, label=lab(name), dchi2_data=d_obs); rows.append(r)
tcs = pd.DataFrame(rows)
if len(tcs):
    tcs["p_cv_holm"] = ds.holm(tcs.p_cv); tcs["p_dent_holm"] = ds.holm(tcs.p_dent)
    ds.show_table(tcs[["label", "n", "dchi2_data", "mean_cv", "sd_cv", "mean_dent", "sd_dent", "p_cv", "p_cv_holm", "p_dent", "p_dent_holm"]],
                  "Tier C: Delta chi2 = chi2(theta=0) - chi2(theta=1). p_CV = P(dchi2 >= data | truth CV); p_DENT = P(dchi2 <= data | truth DENT)")
    ds.plot_dchi2(tcs, savepath=FIG / "tierC_dchi2.png"); ds.show_figure(FIG / "tierC_dchi2.png", "Tier C: data Delta chi2 against the fake-data expectations.")
else:
    print("no Tier C throws available yet")

In [ ]:
import matplotlib.pyplot as plt
if len(tcs):
    n = len(tcs); ncol = 4; nrow = int(np.ceil(n / ncol)); fig, axes = plt.subplots(nrow, ncol, figsize=(4.2 * ncol, 3.3 * nrow)); axes = np.atleast_1d(axes).ravel()
    for ax, (_, r) in zip(axes, tcs.iterrows()):
        ds.plot_tierC_hist(tc[tc.study == r.study], r.dchi2_data, r.label, ax=ax)
    for ax in axes[n:]: ax.axis("off")
    axes[0].legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG / "tierC_hists.png", dpi=170, bbox_inches="tight")
    ds.show_figure(FIG / "tierC_hists.png", "Tier C: distributions of Delta chi2 for fake data from CV (blue) and from CV + DENT (orange); the black line is the data.")

## 9. Regenerated detector-access studies (baseline)
The full set of detector-access studies was regenerated for SBND (40 studies, 111 channel results) and ICARUS Run 2 (28 studies, 72 channel results):
cvrw plus `cvwgt` on the main MC, the SBND data veto added, area-normalised and shape-only. The tables give the production-CV chi-square and p-value.
* **SBND:** 44 of 111 channel results have p < 0.05 and 26 have p < 1e-3. The worst are the primary-score observables (muon primary score, p down to 1e-38).
  Adding DENT at full strength changes any channel's chi-square by between -3.1 and +5.8 units (median +0.19), so none of these failures is
  a DENT effect.
* **ICARUS Run 2:** 14 of 72 have p < 0.05 and 5 have p < 1e-3. ICARUS has no DENT variation, so these are a baseline only.
* These p-values include the production systematic covariance but not any adjustment for the number of studies, and the channels within a study share events.

In [ ]:
base = res[(res.tier == "baseline") & res.variant.isin(["v1", "exDENT", "inj1p0"])]
rows = []
for (det, study, ch), g in base.groupby(["det", "study", "channel"]):
    d = g.drop_duplicates("variant", keep="last").set_index("variant"); r = dict(detector=det, study=study, channel=ch, ndf=int(d.loc["v1", "ndf"]), chi2=d.loc["v1", "chi2"], p=d.loc["v1", "p"])
    if "inj1p0" in d.index: r["chi2 CV+DENT"] = d.loc["inj1p0", "chi2"]; r["dchi2 (0 vs 1)"] = d.loc["v1", "chi2"] - d.loc["inj1p0", "chi2"]
    if "exDENT" in d.index: r["chi2 no DENT cov"] = d.loc["exDENT", "chi2"]
    rows.append(r)
bt = pd.DataFrame(rows)
ds.show_table(bt[bt.detector == "sbnd_run1"].sort_values("p"), "Regenerated SBND detector-access studies (shape-only), sorted by p-value")
ds.show_table(bt[bt.detector == "icarus_run2"].sort_values("p"), "Regenerated ICARUS Run 2 detector-access studies (shape-only), sorted by p-value")

## 10. Limitations and what was not done
* **No energy.** Energy and momentum quantities are blinded and embargoed for SBND data, so nothing here tests DENT on the energy distributions, which is where its uncertainty matters. A study of the high-energy region would need to be run by someone cleared to look.
* **Power.** Single observables have d' of about 0.5 to 1. The set of observables shares events and cannot be combined into one test without a joint covariance, which was not built.
* **Detvar CV is not the production nominal.** Tier A chi-squares mix DENT with other data/MC differences. The Tier B work removes this by using the production CV and the DENT ratio.
* **DENT enters as a one-sided ratio.** The ratio is taken from the detvar CV/DENT pair and applied to the production CV. Whether that is equivalent to using the DENT sample directly was checked
  for the native PROfit detvar mode (unmatched-event mode agrees); a Tier B run with the native detvar mode was not done.
* **Cathode comparisons are hypothesis-prompted** (the analyst had seen them before the plan was written).
* **Trigger emulation.** The flash-PE cut is applied to the MC only. Its efficiency is not part of the comparison.
* **Not examined:** WireMod Y-Z (SBND uses MattMod Y-Z in the systematics), the cathode-region pile-up with an endpoint cut applied to the data comparison, and any ICARUS DENT-like variation.

## 11. Deviations from the plan (all dated in `docs/dent_study/PREREGISTRATION.md`)
* Removed three multiplicity observables (pion, photon, electron) that the selection forces to zero (24 to 21 observables). None was primary.
* The first exploratory data dumps wrongly applied the MC flash cut to data. They were re-made without it; the headline cathode numbers moved by less than 0.1 sigma.
* After the first Tier C result (`crosser_flag`), a third truth (half DENT) and two-sided p-values were added. This is a third fixed point, not a fit.
* Found that two crossing-split studies register DENT under per-population names, so injections were silently ignored. The driver now detects this; both were re-run and the report audits for any inert injection (none remain).
* Replaced Poisson-only Tier C by pseudo-experiments, then by the fixed-covariance replica, after finding that PROfit's injection re-derives the covariance (section 7).
* Changes after seeing results are listed so that the p-values above can be read with the right amount of caution.